# SEC pipeline: filings -> validated claims
Run the cells top to bottom. **Every step is safe to re-run**: finished work is skipped.

One-time setup: keep these files together in the project folder: `fetch_sec_filings.py`, `split_sections.py`, `chunk_sections.py`, `extract_claims.py`, `make_audit_sheet.py`, `company_metadata.csv`, plus Person 1's shared code in `person1export/` (used by step 4 for the team validation rules). Put your name + email in `USER_AGENT` inside `fetch_sec_filings.py`. Your Gemini key stays in the Colab Secrets panel (name must match `SECRET_NAME` in `extract_claims.py`).

Settings you will actually touch live at the top of each `.py` file: companies and filings per company in `fetch_sec_filings.py`; model, `ONLY_TICKERS`, `DROP_RELATIONS` in `extract_claims.py`.

In [ ]:
!pip install -q requests beautifulsoup4 google-genai pycountry rapidfuzz pydantic
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = "/content/drive/MyDrive/YOUR_FOLDER/semiconductor_kg"
os.chdir(ROOT)
print(sorted(f for f in os.listdir('.') if f.endswith(('.py', '.csv'))))

## Step 1: download filings
Reads `company_metadata.csv`, looks up each company on EDGAR, downloads the newest 10-K / 20-F filings. Writes `data/raw/sec/sec_documents.jsonl` and **`sec_coverage.csv`** (which companies the SEC source cannot cover).

In [ ]:
%run fetch_sec_filings.py

## Step 2: find the relevant sections
HTML -> text -> Item 1 (10-K) / Item 4 (20-F). Filings where the split fails are flagged `SUSPICIOUSLY SHORT` and skipped later.

In [ ]:
%run split_sections.py

## Step 3: cut sections into chunks
Produces `data/interim/sec_chunks.jsonl`.

In [ ]:
%run chunk_sections.py

## Step 4: extract and validate claims with the LLM
Only companies in `ONLY_TICKERS` are sent to the model. Raw model responses are saved, so re-running costs no quota for chunks already done. Output: `data/processed/sec_claims.jsonl` (what Person 3 reads) plus rejected claims and provenance.

In [ ]:
import importlib, extract_claims as ec
importlib.reload(ec)
ec.ONLY_TICKERS = {"NONE"}
ec.main()

In [ ]:
import json, collections
from pathlib import Path
cl = [json.loads(l) for l in Path("data/processed/sec_claims.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
pv = [c for c in cl if c["relation"] == "PROVIDES"]
print("PROVIDES object types:", dict(collections.Counter(c["object_type"] for c in pv)))
for t in ("Input", "Product", "Technology"):
    print(t, "->", [c["object_mention"] for c in pv if c["object_type"] == t][:8])

## Quick health report
Counts, rejection reasons, warnings, and the strong flags (a name that appears nowhere in the text it was extracted from).

In [ ]:
import json, collections
from pathlib import Path
P = Path('data/processed')
rd = lambda f: [json.loads(l) for l in (P / f).read_text(encoding='utf-8').splitlines() if l.strip()]
claims, prov, rej = rd('sec_claims.jsonl'), rd('sec_claims_provenance.jsonl'), rd('sec_claims_rejected.jsonl')
by_id = {c['claim_id']: c for c in claims}
print(len(claims), 'claims kept |', len(rej), 'rejected')
print('rejection reasons:', dict(collections.Counter(x for r in rej for x in r['reasons'])))
print('relations:', dict(collections.Counter(c['relation'] for c in claims)))
print('warnings:', dict(collections.Counter(w for p in prov for w in p['warnings'])))
print('\nstrong flags:')
for p in prov:
    if any(w.endswith('_not_in_chunk') for w in p['warnings']):
        c = by_id[p['claim_id']]
        print(' ', c['subject_mention'], '--', c['relation'], '->', c['object_mention'])

## Step 5 (optional): sampling sheet for the human audit
Writes `data/processed/sec_audit_sheet_seed7.csv`; fill the `label` column in Excel.

In [ ]:
%run make_audit_sheet.py